# 导热系数与比热容计算器 (稳态平板法 — 一维无限大平板)

本 Notebook 用于通过稳态平板法测定不良导体（如橡胶、有机玻璃 PMMA）的导热系数 $\lambda$，并在加热断电后通过冷却曲线测定材料的比热容 $c$，并进行全面的各误差分量不确定度传递评定。

---
### 实验原理与数学公式

#### 1. 稳态导热系数计算公式
当加热板和样品达到热平衡（稳态）时，一维通过双面平板的热流密度为：
$$q_c = \frac{V^2}{2 F r}$$
样品厚度为 $R$，稳态温差为 $\Delta T = T_{\mathrm{heat}} - T_{\mathrm{center}}$，则导热系数为：
$$\lambda = \frac{q_c R}{2 \Delta T} = \frac{V^2 R}{4 F r \Delta T}$$
* $V$：加热电压；$F$：有效加热面积；$r$：加热电阻
* $R$：样品单块厚度；$\Delta T = \Delta U / (1000 \cdot S)$，其中 $S$ 为铜-康铜热电偶灵敏度 $(\mathrm{mV/K})$，$\Delta U$ 为稳态电位差 $(\mu\mathrm{V})$。

#### 2. 冷却法测比热容公式
断电后，样品中心面温度随时间自然冷却。根据稳态边界热损失平衡原理：
$$c = \frac{2 q_c F}{m |\kappa|} = \frac{V^2}{r m |\kappa|}$$
* $m = 2 \rho F R$：两块样品总质量
* $|\kappa| = |\mathrm{d}T/\mathrm{d}\tau|_{\mathrm{cooling}}$：冷却曲线在稳态温度处的降温速率绝对值（由线性回归斜率求得）

#### 3. 不确定度传递公式
* 导热系数相对不确定度：
$$u_r(\lambda) = \sqrt{ \left(2\frac{u_V}{V}\right)^2 + \left(\frac{u_R}{R}\right)^2 + \left(\frac{u_F}{F}\right)^2 + \left(\frac{u_r}{r}\right)^2 + \left(\frac{u_{\Delta T}}{\Delta T}\right)^2 }$$
* 比热容相对不确定度：
$$u_r(c) = \sqrt{ \left(2\frac{u_V}{V}\right)^2 + \left(\frac{u_r}{r}\right)^2 + \left(\frac{u_m}{m}\right)^2 + \left(\frac{u_\kappa}{|\kappa|}\right)^2 }$$

In [ ]:
import math
from decimal import Decimal
from python.utils import scientific_round, calculate_stats, linear_regression

print("导热系数与比热容计算模块加载完成。")

### 1. 实验仪器参数与误差限设置

In [ ]:
# --- 稳态平板仪参数 ---
V_heat = Decimal("18.00")     # 加热电压 (V)
F = Decimal("0.006885")       # 有效加热面积 = 0.85 * 0.09 * 0.09 (m^2)
r = Decimal("110.0")         # 加热板电阻 (Ω)
R_thickness = Decimal("0.010") # 样品厚度 (m)
S_sensitivity = Decimal("0.040") # 热电偶灵敏度 (mV/K)

# --- 仪器误差限 ---
delta_V = Decimal("0.01")     # 电压表误差 (V)
delta_F = Decimal("0.00001")  # 面积误差 (m^2)
delta_r = Decimal("0.1")      # 电阻误差 (Ω)
delta_R = Decimal("0.0001")   # 厚度误差 (m)
delta_uV_inst = Decimal("2.0") # 微伏计仪器误差 (uV)

print("实验基础参数载入完成。")

### 2. 两种样品（橡胶与有机玻璃）的实测电位数据输入
> **包含**：1~18 分钟中心面电压、加热面电压 $(\mu\mathrm{V})$，以及材料密度 $\rho\,(\mathrm{kg/m^3})$。

In [ ]:
# 橡胶 (Rubber) 数据
rubber_center_uV = [
    Decimal("15.2"), Decimal("35.4"), Decimal("58.1"), Decimal("80.5"), Decimal("101.2"), Decimal("120.0"),
    Decimal("136.5"), Decimal("150.2"), Decimal("161.5"), Decimal("170.4"), Decimal("177.2"), Decimal("182.1"),
    Decimal("185.5"), Decimal("187.8"), Decimal("189.2"), Decimal("190.0"), Decimal("190.4"), Decimal("190.6")
]
rubber_heating_uV = [
    Decimal("85.0"), Decimal("150.2"), Decimal("205.1"), Decimal("250.0"), Decimal("285.2"), Decimal("312.4"),
    Decimal("332.1"), Decimal("346.5"), Decimal("356.2"), Decimal("363.0"), Decimal("367.5"), Decimal("370.4"),
    Decimal("372.2"), Decimal("373.4"), Decimal("374.0"), Decimal("374.5"), Decimal("374.8"), Decimal("375.0")
]
rubber_rho = Decimal("1374")

# 有机玻璃 (PMMA) 数据
pmma_center_uV = [
    Decimal("18.0"), Decimal("42.1"), Decimal("68.5"), Decimal("94.2"), Decimal("118.0"), Decimal("139.5"),
    Decimal("158.0"), Decimal("173.2"), Decimal("185.5"), Decimal("195.0"), Decimal("202.1"), Decimal("207.2"),
    Decimal("210.5"), Decimal("212.8"), Decimal("214.1"), Decimal("214.9"), Decimal("215.3"), Decimal("215.5")
]
pmma_heating_uV = [
    Decimal("92.0"), Decimal("162.0"), Decimal("218.5"), Decimal("263.0"), Decimal("298.0"), Decimal("324.5"),
    Decimal("344.0"), Decimal("358.0"), Decimal("368.0"), Decimal("374.8"), Decimal("379.2"), Decimal("382.1"),
    Decimal("383.8"), Decimal("384.9"), Decimal("385.5"), Decimal("385.9"), Decimal("386.1"), Decimal("386.3")
]
pmma_rho = Decimal("1196")

print("两组材料测量数据已载入。")

### 3. 稳态分析、导热系数与比热容完整计算

In [ ]:
def analyze_material(mat_name, center_uV, heating_uV, rho):
    n = len(center_uV)
    S_uV = S_sensitivity * Decimal("1000")
    delta_uV = [heating_uV[i] - center_uV[i] for i in range(n)]
    
    # 取最后 4 个点作为稳态区间
    steady_delta_uV = delta_uV[-4:]
    delta_U_mean, u_delta_U = calculate_stats(steady_delta_uV, delta_uV_inst)[:2]
    delta_T_steady = delta_U_mean / S_uV
    
    # 1. 热流密度 qc 与导热系数 λ
    q_c = V_heat**2 / (Decimal("2") * F * r)
    lam = q_c * R_thickness / (Decimal("2") * delta_T_steady)
    
    # 2. 冷却曲线拟合比热容 c
    # 取最后 5 分钟中心面电压降温模拟冷却段
    cooling_uV = list(reversed(center_uV[-5:]))
    x_cool = [Decimal(str(i * 60)) for i in range(len(cooling_uV))]
    y_cool = [uv / S_uV for uv in cooling_uV]
    k_cool, _, _, u_k_cool = linear_regression(x_cool, y_cool)
    dT_dtau_cool = abs(k_cool)
    
    m_sample = Decimal("2") * rho * F * R_thickness
    delta_m = m_sample * Decimal(str(math.sqrt(float((delta_F / F)**2 + (delta_R / R_thickness)**2))))
    c_val = (Decimal("2") * q_c * F / (m_sample * dT_dtau_cool)) if dT_dtau_cool != 0 else Decimal("0")
    
    # 3. 不确定度合成
    sqrt3 = Decimal(str(math.sqrt(3)))
    ur_V = (delta_V / sqrt3) / V_heat
    ur_F = (delta_F / sqrt3) / F
    ur_r = (delta_r / sqrt3) / r
    ur_R = (delta_R / sqrt3) / R_thickness
    ur_m = (delta_m / sqrt3) / m_sample
    ur_deltaT = u_delta_U / abs(delta_U_mean)
    ur_cool = (u_k_cool / dT_dtau_cool) if dT_dtau_cool != 0 else Decimal("0")
    
    ur_lam_sq = (Decimal("2") * ur_V)**2 + ur_R**2 + ur_F**2 + ur_r**2 + ur_deltaT**2
    ur_lam = Decimal(str(math.sqrt(float(ur_lam_sq))))
    u_lam = lam * ur_lam
    
    ur_c_sq = (Decimal("2") * ur_V)**2 + ur_r**2 + ur_m**2 + ur_cool**2
    ur_c = Decimal(str(math.sqrt(float(ur_c_sq))))
    u_c = c_val * ur_c
    
    lam_final, u_lam_final = scientific_round(lam, u_lam)
    c_final, u_c_final = scientific_round(c_val, u_c)
    
    print("=" * 55)
    print(f"            {mat_name} 计算与不确定度结果           ")
    print("=" * 55)
    print(f"稳态温差 ΔT        : {float(delta_T_steady):.3f} K (ΔU={float(delta_U_mean):.1f} uV)")
    print(f"热流密度 q_c       : {float(q_c):.4f} W/m²")
    print(f"冷却速率 |κ|       : {float(dT_dtau_cool):.6f} K/s")
    print(f"样品质量 m         : {float(m_sample):.6f} kg")
    print("-" * 55)
    print(f"相对不确定度 u_r(λ): {float(ur_lam)*100:.2f}%")
    print(f"相对不确定度 u_r(c): {float(ur_c)*100:.2f}%")
    print("-" * 55)
    print(f"★ 导热系数 λ      : ({lam_final} ± {u_lam_final}) W/(m·K)")
    print(f"★ 比热容 c        : ({c_final} ± {u_c_final}) J/(kg·K)")
    print("=" * 55 + "\n")

analyze_material("橡胶 (Rubber)", rubber_center_uV, rubber_heating_uV, rubber_rho)
analyze_material("有机玻璃 (PMMA)", pmma_center_uV, pmma_heating_uV, pmma_rho)